# 01 - PDF Extraction & Structure Parsing

**Task 1, Fase 1:** Mengekstrak teks dari PDF UU, mem-parse struktur hierarki dokumen, dan membuat chunks untuk LLM processing.

**Pipeline:** `PDF → Extract Text → Parse Structure → Create Chunks`

**Chunking Strategies:**
- **Structure-Aware** (recommended): Groups by BAB, injects hierarchy context headers, splits at Pasal boundaries
- **Naive**: Sliding window with token overlap (legacy)

In [1]:
# === Setup ===
import sys
import json
import os
from pathlib import Path

# Ensure project root is in path
PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF files available: {[p.name for p in Path('data/raw').glob('*.pdf')]}")

Project root: d:\TA\llm-driven-legal-kg-visualization
PDF files available: ['Permen Kominfo Nomor 5 Tahun 2017.pdf', 'Perpres Nomor 132 Tahun 2022.pdf', 'Perpres Nomor 95 Tahun 2018.pdf', 'POJK 11 - 03 - 2022.pdf', 'PP Nomor 71 Tahun 2019.pdf', 'PP Nomor 80 Tahun 2019.pdf', 'PP Nomor 82 Tahun 2012.pdf', 'UU Nomor  19 Tahun 2016.pdf', 'UU Nomor 11 Tahun 2008.pdf', 'UU Nomor 27 Tahun 2022.pdf', 'UU Nomor 36 Tahun 1999.pdf']


## Step 1: Extract Text from PDF

Menggunakan PyMuPDF (digital text extraction) dengan PaddleOCR fallback untuk halaman scanned.

In [2]:
from pipeline.extract.pdf_extractor import extract_pdf, save_extracted_document

# === CONFIGURE: Choose PDF file ===
PDF_PATH = "data/raw/POJK 11 - 03 - 2022.pdf"

# Extract PDF
doc = extract_pdf(PDF_PATH)

# Save to JSON
output_path = save_extracted_document(doc, "data/extracted")

print(f"\n=== Extraction Summary ===")
print(f"Document ID : {doc.document_id}")
print(f"Title       : {doc.title}")
print(f"UU Number   : {doc.uu_number}")
print(f"Year        : {doc.year}")
print(f"Total Pages : {doc.total_pages}")
print(f"Saved to    : {output_path}")

# Count scanned vs digital pages
scanned = sum(1 for p in doc.pages if p['is_scanned'])
digital = doc.total_pages - scanned
print(f"\nDigital pages: {digital}, Scanned pages: {scanned}")


=== Extraction Summary ===
Document ID : POJK_11_2022
Title       : POJK Nomor 11/2022 tentang PENYELENGGARAAN TEKNOLOGI INFORMASI
UU Number   : 11/2022
Year        : 2022
Total Pages : 76
Saved to    : data/extracted\POJK_11_2022.json

Digital pages: 76, Scanned pages: 0


In [3]:
# Preview first 3 pages
for page in doc.pages[:3]:
    print(f"\n{'='*60}")
    print(f"PAGE {page['page_number']} (scanned={page['is_scanned']})")
    print(f"{'='*60}")
    text = page['clean_text'][:500] if page['clean_text'] else page['selectable_text'][:500]
    print(text)
    print("...")


PAGE 1 (scanned=False)
PERATURAN OTORITAS JASA KEUANGAN
REPUBLIK INDONESIA
NOMOR 11 /POJK.03/2022
TENTANG
PENYELENGGARAAN TEKNOLOGI INFORMASI
OLEH BANK UMUM

DENGAN RAHMAT TUHAN YANG MAHA ESA

DEWAN KOMISIONER OTORITAS JASA KEUANGAN,

Menimbang
: a.
bahwa untuk mendukung kelangsungan operasional
serta pelayanan bank kepada masyarakat, dibutuhkan
pemanfaatan teknologi informasi oleh bank;

b.
bahwa pemanfaatan teknologi informasi berpotensi
meningkatkan eksposur risiko bagi bank sehingga bank
perlu memperkuat tata kelo
...

PAGE 2 (scanned=False)
Informasi oleh Bank Umum sebagaimana telah diubah
dengan
Peraturan
Otoritas
Jasa
Keuangan
Nomor
13/POJK.03/2020
tentang
Perubahan
atas
Peraturan
Otoritas
Jasa
Keuangan
Nomor
38/POJK.03/2016
tentang
Penerapan
Manajemen
Risiko
dalam
Penggunaan
Teknologi
Informasi oleh Bank Umum;

d.
bahwa
berdasarkan
pertimbangan
sebagaimana
dimaksud dalam huruf a, huruf b, dan huruf c, perlu
menetapkan Peraturan Otoritas Jasa Keuangan tentang
Penyelenggaraan Te

## Step 2: Parse Document Structure

Mendeteksi hierarki komponen: BAB → BAGIAN → PARAGRAF → PASAL → AYAT → HURUF → ANGKA

Parser juga mendeteksi bagian **PENJELASAN** secara otomatis dan memberi flag `is_penjelasan=True` pada komponen yang termasuk dalam bagian Penjelasan.

In [4]:
from pipeline.extract.structure_parser import parse_document_structure, save_parsed_document, print_component_tree

# Load extracted document
EXTRACTED_PATH = f"data/extracted/{doc.document_id}.json"
with open(EXTRACTED_PATH, "r", encoding="utf-8") as f:
    extracted_doc = json.load(f)

# Parse structure
components = parse_document_structure(extracted_doc)
output_path = save_parsed_document(extracted_doc['document_id'], components, 'data/parsed')

# Summary
type_counts = {}
for c in components:
    type_counts[c.component_type] = type_counts.get(c.component_type, 0) + 1

penjelasan_count = sum(1 for c in components if c.is_penjelasan)

print(f"\n=== Parsing Summary ===")
print(f"Total Components: {len(components)}")
print(f"By Type:")
for comp_type, count in sorted(type_counts.items()):
    print(f"  {comp_type:12s}: {count}")
print(f"\nPenjelasan components: {penjelasan_count} (will be filtered in structure-aware chunking)")
print(f"Saved to: {output_path}")

[INFO] Penjelasan detected at line 2534 (page 44)

=== Parsing Summary ===
Total Components: 177
By Type:
  BAB         : 14
  BAGIAN      : 17
  PASAL       : 146

Penjelasan components: 73 (will be filtered in structure-aware chunking)
Saved to: data/parsed\POJK_11_2022.json


In [5]:
# Print document hierarchy tree (depth=2 for readability)
print("=== Document Hierarchy (depth=2) ===")
print_component_tree(components, max_depth=2)

=== Document Hierarchy (depth=2) ===
[BAB] I - KETENTUAN UMUM
  ├── [PASAL] 1 | Dalam Peraturan Otoritas Jasa Keuangan ini, yang dimaksud de...
[BAB] II - TATA KELOLA TI BANK
  ├── [BAGIAN] Kesatu - Umum
    ├── [PASAL] 2 | (1) Bank wajib menerapkan tata kelola TI yang baik dalam pen...
    ├── [PASAL] 3 | (1) Dalam menerapkan tata kelola TI, Bank wajib melakukan pe...
  ├── [BAGIAN] Kedua - Penerapan Tata Kelola TI Bank
    ├── [PASAL] 4 | Bank wajib menetapkan wewenang dan tanggung jawab yang jelas...
    ├── [PASAL] 5 | Wewenang dan tanggung jawab Direksi sebagaimana dimaksud dal...
    ├── [PASAL] 6 | Wewenang dan tanggung jawab Dewan Komisaris sebagaimana dima...
    ├── [PASAL] 7 | (1) Bank wajib memiliki komite pengarah TI. (2) Komite penga...
    ├── [PASAL] 8 | (1) Bank wajib memiliki satuan kerja penyelenggara TI yang b...
    ├── [PASAL] 9 | (1) Bank yang melanggar ketentuan sebagaimana dimaksud dalam...
    ├── [PASAL] 10 | Ketentuan lebih lanjut mengenai penerapan tata kel

## Step 3: Create Chunks

Pilih salah satu strategy:

### Strategy A: Structure-Aware (Recommended)
- Groups komponen per BAB
- Inject context header `[DOKUMEN]`, `[BAB]`, `[BAGIAN]` di setiap chunk
- Split di batas Pasal (tidak memotong mid-article)
- Filter Penjelasan otomatis (via `is_penjelasan` flag dari parser)

### Strategy B: Naive (Legacy)
- Sliding window dengan token overlap
- Tanpa context injection
- Cocok untuk ablation study / perbandingan

In [6]:
# === CONFIGURE: Choose chunking strategy ===
CHUNKING_STRATEGY = "structure_aware"  # Options: "structure_aware" or "naive"
MAX_TOKENS = 3000  # Maximum tokens per chunk (for structure_aware)
INCLUDE_PENJELASAN = False  # Set True to include Penjelasan sections

In [7]:
from pipeline.extract.chunker import create_chunks, create_structure_aware_chunks, save_chunks

# Load parsed document
PARSED_PATH = f"data/parsed/{doc.document_id}.json"
with open(PARSED_PATH, "r", encoding="utf-8") as f:
    parsed_doc = json.load(f)

# Create chunks based on selected strategy
if CHUNKING_STRATEGY == "structure_aware":
    chunks = create_structure_aware_chunks(
        parsed_doc['components'],
        parsed_doc['document_id'],
        doc_title=parsed_doc.get('title', ''),
        max_tokens=MAX_TOKENS,
        include_penjelasan=INCLUDE_PENJELASAN,
    )
else:
    chunks = create_chunks(
        parsed_doc['components'],
        parsed_doc['document_id'],
        min_tokens=400,
        max_tokens=800,
        overlap_tokens=100
    )

# Save
output_path = save_chunks(
    parsed_doc['document_id'], chunks, 'data/chunks',
    strategy=CHUNKING_STRATEGY
)

# Summary
if chunks:
    token_counts = [c.token_count for c in chunks]
    print(f"\n=== Chunking Summary ===")
    print(f"Strategy     : {CHUNKING_STRATEGY}")
    print(f"Total Chunks : {len(chunks)}")
    print(f"Token Stats  : min={min(token_counts)}, max={max(token_counts)}, avg={sum(token_counts)//len(token_counts)}")
    print(f"Penjelasan   : {'included' if INCLUDE_PENJELASAN else 'excluded'}")
    print(f"Saved to     : {output_path}")


=== Chunking Summary ===
Strategy     : structure_aware
Total Chunks : 24
Token Stats  : min=177, max=2737, avg=852
Penjelasan   : excluded
Saved to     : data/chunks\POJK_11_2022_chunks.json


In [8]:
# Preview first 3 chunks
for chunk in chunks[:3]:
    print(f"\n{'='*60}")
    print(f"Chunk: {chunk.chunk_id}")
    print(f"Tokens: {chunk.token_count}  |  Pages: {chunk.page_range}  |  Parent: {chunk.parent_component_id}")
    print(f"{'='*60}")
    print(chunk.text[:500])
    print("...")


Chunk: POJK_11_2022__struct_chunk_000
Tokens: 837  |  Pages: [3, 4]  |  Parent: POJK_11_2022__BAB_I
[DOKUMEN: ]
[BAB: BAB I KETENTUAN UMUM]
---
Pasal 1
Dalam Peraturan Otoritas Jasa Keuangan ini, yang
dimaksud dengan:
1.
Bank Umum yang selanjutnya disebut sebagai Bank
adalah bank yang melaksanakan kegiatan usaha secara
konvensional atau melaksanakan kegiatan usaha
berdasarkan prinsip syariah, yang dalam kegiatannya
memberikan jasa dalam lalu lintas pembayaran,
termasuk kantor cabang dari bank yang berkedudukan
di luar negeri dan unit usaha syariah.
2.
Teknologi Informasi yang selanjutnya disin
...

Chunk: POJK_11_2022__struct_chunk_001
Tokens: 802  |  Pages: [4, 5, 6]  |  Parent: POJK_11_2022__BAB_II
[DOKUMEN: ]
[BAB: BAB II TATA KELOLA TI BANK]
[BAGIAN: Bagian Kesatu Umum]
---
Pasal 2
(1)
Bank wajib menerapkan tata kelola TI yang baik dalam
penyelenggaraan TI.
(2)
Dalam
menerapkan
tata
kelola
TI
yang
baik
sebagaimana
dimaksud
pada
ayat
(1),
Bank
mempertimbangkan faktor paling sedikit

## (Optional) Batch Process All PDFs

Jalankan cell berikut untuk memproses semua PDF sekaligus.

In [9]:
# from pipeline.extract.pdf_extractor import extract_pdf, save_extracted_document
# from pipeline.extract.structure_parser import parse_document_structure, save_parsed_document
# from pipeline.extract.chunker import create_structure_aware_chunks, save_chunks

# pdf_files = sorted(Path('data/raw').glob('*.pdf'))
# print(f"Found {len(pdf_files)} PDF files\n")

# for pdf_path in pdf_files:
#     print(f"--- {pdf_path.name} ---")
    
#     # Step 1: Extract
#     doc = extract_pdf(str(pdf_path))
#     save_extracted_document(doc, 'data/extracted')
    
#     # Step 2: Parse
#     with open(f'data/extracted/{doc.document_id}.json', 'r', encoding='utf-8') as f:
#         extracted = json.load(f)
#     components = parse_document_structure(extracted)
#     save_parsed_document(doc.document_id, components, 'data/parsed')
    
#     # Step 3: Chunk (structure-aware)
#     with open(f'data/parsed/{doc.document_id}.json', 'r', encoding='utf-8') as f:
#         parsed = json.load(f)
#     chunks = create_structure_aware_chunks(
#         parsed['components'], parsed['document_id'],
#         doc_title=parsed.get('title', ''),
#         max_tokens=MAX_TOKENS,
#         include_penjelasan=INCLUDE_PENJELASAN,
#     )
#     save_chunks(parsed['document_id'], chunks, 'data/chunks', strategy='structure_aware')
    
#     penj = sum(1 for c in components if c.is_penjelasan)
#     tokens = [c.token_count for c in chunks]
#     print(f"  {doc.document_id}: {len(components)} components ({penj} penjelasan), "
#           f"{len(chunks)} chunks (avg {sum(tokens)//max(len(tokens),1)} tokens)\n")

## Summary

Fase 1 selesai! Output tersimpan di:
- `data/extracted/{doc_id}.json` — Raw extracted text per page
- `data/parsed/{doc_id}.json` — Hierarchical structure (BAB → PASAL → ...) with `is_penjelasan` flags
- `data/chunks/{doc_id}_chunks.json` — Chunks ready for LLM extraction

**Next:** Run `02_llm_extraction.ipynb` to extract Knowledge Graph triples using Gemini.